In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Estimator 입력 및 출력

<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
이 페이지에서는 IBM Quantum&reg; 컴퓨팅 자원에서 워크로드를 실행하는 Qiskit Runtime Estimator Primitive의 입력 및 출력에 대한 개요를 제공합니다. Estimator는 [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs)이라는 데이터 구조를 사용하여 벡터화된 워크로드를 효율적으로 정의할 수 있도록 합니다. PUB는 Estimator Primitive의 [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) 메서드에 입력으로 사용되며, 정의된 워크로드를 Job으로 실행합니다. Job이 완료된 후에는 사용한 PUB 및 Primitive에서 지정한 런타임 옵션에 따라 결과가 반환됩니다.
## 입력
각 PUB는 다음 형식을 따릅니다:

(`<단일 Circuit>`, `<하나 이상의 Observable>`, `<선택적: 하나 이상의 매개변수 값>`, `<선택적: 정밀도>`),

선택적인 `매개변수 값`은 목록 또는 단일 매개변수가 될 수 있습니다. Observable과 매개변수 값의 요소는 [Primitive 입력 및 출력](primitive-input-output#broadcasting-rules) 항목에 설명된 NumPy 브로드캐스팅 규칙에 따라 결합되며, 브로드캐스트된 형태의 각 요소에 대해 하나의 기댓값 추정치가 반환됩니다.

> **Note:** 입력에 측정이 포함된 경우 이는 무시됩니다.

Estimator Primitive의 경우 PUB에는 최대 네 가지 값이 포함될 수 있습니다:
- 하나 이상의 [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) 객체를 포함할 수 있는 단일 `QuantumCircuit`
- 추정할 기댓값을 지정하는 하나 이상의 Observable 목록으로, 배열로 정렬됩니다(예: 단일 Observable은 0차원 배열, Observable 목록은 1차원 배열 등). 데이터는 `Pauli`, `SparsePauliOp`, `PauliList`, `str`과 같은 `ObservablesArrayLike` 형식 중 하나로 제공될 수 있습니다.
   > **Note:** - **동일한 PUB 내의** 가환 Observable은 [이 메서드](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting)를 사용하여 그룹화됩니다.
>    - 다른 PUB에 있는 가환 Observable은 동일한 Circuit을 가지더라도 동일한 측정을 사용하여 추정되지 않습니다. 각 PUB는 서로 다른 측정 기저를 나타내므로 각 PUB에 대해 별도의 측정이 필요합니다.
>    - 가환 Observable이 동일한 측정을 사용하여 추정되도록 하려면 동일한 PUB 내에 그룹화하세요.
- Circuit에 바인딩할 매개변수 값의 컬렉션. 이는 마지막 인덱스가 Circuit의 `Parameter` 객체에 해당하는 단일 배열 형식 객체로 지정하거나, Circuit에 `Parameter` 객체가 없는 경우 생략(또는 동등하게 `None`으로 설정)할 수 있습니다.
- (선택적) 추정할 기댓값의 목표 정밀도
---

다음 코드는 `Estimator` Primitive에 대한 벡터화된 입력 예제를 보여주고, 이를 IBM&reg; Backend에서 단일 `RuntimeJobV2` 객체로 실행합니다.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## 출력
하나 이상의 PUB가 QPU로 전송되어 Job이 성공적으로 완료되면, 데이터는 `RuntimeJobV2.result()` 메서드를 호출하여 접근하는 [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) 컨테이너 객체로 반환됩니다.

`PrimitiveResult`는 각 PUB의 실행 결과를 포함하는 [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) 객체의 반복 가능한 목록을 포함합니다.

이 목록의 각 요소는 Primitive의 `run()` 메서드에 제출된 각 PUB에 해당합니다(예: 20개의 PUB와 함께 제출된 Job은 각 PUB에 해당하는 20개의 `PubResult` 객체 목록을 포함하는 `PrimitiveResult` 객체를 반환합니다).

Estimator Primitive의 각 `PubResult`는 최소한 기댓값 배열(`PubResult.data.evs`)과 관련 표준 편차(사용된 `resilience_level`에 따라 `PubResult.data.stds` 또는 `PubResult.data.ensemble_standard_error`)를 포함하지만, 지정된 오류 완화 옵션에 따라 더 많은 데이터를 포함할 수 있습니다.

각 `PubResult` 객체는 `data`와 `metadata` 속성을 모두 가집니다.
    - `data` 속성은 실제 측정값, 표준 편차 등을 포함하는 사용자 정의 [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin)입니다.
    - `DataBin`은 관련 PUB의 형태나 구조 및 Job을 제출하는 데 사용된 Primitive에서 지정한 오류 완화 옵션(예: [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) 또는 [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec))에 따라 다양한 속성을 가집니다.
    - `metadata` 속성에는 사용된 런타임 및 오류 완화 옵션에 대한 정보가 포함됩니다(이 페이지의 [결과 메타데이터](#result-metadata) 섹션에서 나중에 설명됩니다).

다음은 Estimator 출력에 대한 `PrimitiveResult` 데이터 구조의 시각적 개요입니다:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

간단히 말해서, 단일 Job은 `PrimitiveResult` 객체를 반환하고 하나 이상의 `PubResult` 객체 목록을 포함합니다. 이 `PubResult` 객체들은 Job에 제출된 각 PUB의 측정 데이터를 저장합니다.

아래 코드 스니펫은 위에서 생성된 Job의 `PrimitiveResult`(및 관련 `PubResult`) 형식을 설명합니다.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUBs and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
